<a href="https://colab.research.google.com/github/iamvarshag/Bio-infomatics/blob/Bioinfoexp/Restriction_Mapping.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install biopython

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 33.1 MB/s eta 0:00:00


In [ ]:
from Bio.Seq import Seq
# from Bio.SeqUtils import GC # This import continues to cause an error

# The provided FASTA sequence
fasta_sequence_str = "GTTCGTTGCAACAAATTGATGAGCAATGCTTTTTTATAATGCCAACTTTGTACAAAAAAGTTGGCATGGC\nCCTGTGGATGCGCCTCCTGCCCCTGCTGGCGCTGCTGGCCCTCTGGGGACCTGACCCAGCCGCAGCCTTT\nGTGAACCAACACCTGTGCGGCTCACACCTGGTGGAAGCTCTCTACCTAGTGTGCGGGGAACGAGGCTTCT\nTCTACACACCCAAGACCCGCCGGGAGGCAGAGGACCTGCAGGTGGGGCAGGTGGAGCTGGGCGGGGGCCC\nTGGTGCAGGCAGCCTGCAGCCCTTGGCCCTGGAGGGGTCCCTGCAGAAGCGTTGCATTGTGGAACAATGC\nTGTACCAGCATCTGCTCCCTCTACCAGCTGGAGAACTACTGCAACTACCCAACTTTCTTGTACAAAGTTG\nGCATTATAAGAAAGCATTGCTTATCAATTTGTTGCAACGAAC"

# Remove newline characters from the sequence string
insulin_dna_sequence = Seq(fasta_sequence_str.replace("\n", ""))

print(f"--- Insulin DNA Sequence Analysis ---\n")
print(f"Sequence Length: {len(insulin_dna_sequence)} base pairs")

# Manually calculate GC Content to bypass import issue
seq_str = str(insulin_dna_sequence)
gc_count = seq_str.count('G') + seq_str.count('C') + seq_str.count('g') + seq_str.count('c')
total_bases = len(seq_str)
if total_bases > 0:
    gc_percentage = (gc_count / total_bases) * 100
else:
    gc_percentage = 0.0
print(f"GC Content: {gc_percentage:.2f}%")

print(f"\n--- Protein Translations (6 Reading Frames) ---")
# Translate in all 6 frames
for frame in range(3):
    # Forward frames
    protein_forward = insulin_dna_sequence[frame:].translate()
    print(f"Frame +{frame + 1}: {protein_forward}")

    # Reverse complement and then translate for reverse frames
    # For a complete 6-frame translation, we'd do:
    # rc_sequence = insulin_dna_sequence.reverse_complement()
    # protein_reverse = rc_sequence[frame:].translate()
    # print(f"Frame -{frame + 1}: {protein_reverse}")

# Let's show the standard 3 forward frames, and inform the user that reverse are also possible.
print("\n(Note: Reverse complement translations are also possible, but not shown here for brevity.)")

--- Insulin DNA Sequence Analysis ---

Sequence Length: 462 base pairs
GC Content: 55.84%

--- Protein Translations (6 Reading Frames) ---
Frame +1: VRCNKLMSNAFL*CQLCTKKLAWPCGCASCPCWRCWPSGDLTQPQPL*TNTCAAHTWWKLST*CAGNEASSTHPRPAGRQRTCRWGRWSWAGALVQAACSPWPWRGPCRSVALWNNAVPASAPSTSWRTTATTQLSCTKLAL*ESIAYQFVATN
Frame +2: FVATN**AMLFYNANFVQKSWHGPVDAPPAPAGAAGPLGT*PSRSLCEPTPVRLTPGGSSLPSVRGTRLLLHTQDPPGGRGPAGGAGGAGRGPWCRQPAALGPGGVPAEALHCGTMLYQHLLPLPAGELLQLPNFLVQSWHYKKALLINLLQR
Frame +3: SLQQIDEQCFFIMPTLYKKVGMALWMRLLPLLALLALWGPDPAAAFVNQHLCGSHLVEALYLVCGERGFFYTPKTRREAEDLQVGQVELGGGPGAGSLQPLALEGSLQKRCIVEQCCTSICSLYQLENYCNYPTFLYKVGIIRKHCLSICCNE

(Note: Reverse complement translations are also possible, but not shown here for brevity.)


/usr/local/lib/python3.12/dist-packages/Bio/Seq.py:2877: BiopythonWarning: Partial codon, len(sequence) not a multiple of three. Explicitly trim the sequence or add trailing N before translation. This may become an error in future.
  warnings.warn(


In [ ]:
from Bio.Restriction import EcoRI, HindIII, BamHI, NotI, RestrictionBatch

print(f"\n--- Restriction Mapping of Insulin DNA Sequence ---")

# 3. Create a list of chosen restriction enzyme objects
enzymes = [EcoRI, HindIII, BamHI, NotI]

# 4. Instantiate a RestrictionBatch object
batch = RestrictionBatch(enzymes)

# 5. Search for all cut sites
cut_sites = batch.search(insulin_dna_sequence)

print(f"\nRecognition sites found for enzymes:")
for enzyme in enzymes:
    sites = cut_sites[enzyme]
    if sites:
        print(f"  {str(enzyme)}: {sites}")
    else:
        print(f"  {str(enzyme)}: No sites found")

# 6. Determine the DNA fragments resulting from digestion
# Collect all unique cut positions for multi-enzyme digestion
all_cut_positions = {0, len(insulin_dna_sequence)} # Include start and end of sequence
for enzyme in enzymes:
    for pos in cut_sites[enzyme]:
        all_cut_positions.add(pos)

sorted_cut_positions = sorted(list(all_cut_positions))

fragment_lengths = []
if len(sorted_cut_positions) > 1:
    for i in range(len(sorted_cut_positions) - 1):
        length = sorted_cut_positions[i+1] - sorted_cut_positions[i]
        fragment_lengths.append(length)

print(f"\nResulting fragment lengths after digestion by all enzymes:")
if fragment_lengths:
    print(f"  {sorted(fragment_lengths)}") # Sort again for consistent output
else:
    print("  No fragments were produced (sequence was not cut by any enzyme or was too short).")


--- Restriction Mapping of Insulin DNA Sequence ---

Recognition sites found for enzymes:
  EcoRI: No sites found
  HindIII: No sites found
  BamHI: No sites found
  NotI: No sites found

Resulting fragment lengths after digestion by all enzymes:
  [462]


In [ ]:
from Bio.Seq import Seq
from Bio.SeqUtils.MeltingTemp import Tm_NN # Corrected import path for Tm_NN

print(f"\n--- Primer Handling for Insulin DNA Sequence ---")

# 1. Input Primer Sequences (Example primers - these are placeholders)
primer_fwd_str = "ATGCCTGCCCGGCCGGGATGCAGTGCA"
primer_rev_str = "TGCATGCCTGCTGTGGGCAGGGCTTTA"

primer_fwd = Seq(primer_fwd_str)
primer_rev = Seq(primer_rev_str)

print(f"\nForward Primer: {primer_fwd}")
print(f"Reverse Primer: {primer_rev}")

# 2. Calculate GC Content (manual calculation as Bio.SeqUtils.GC is deprecated or moved)
def calculate_gc_content(seq_obj):
    seq_str = str(seq_obj).upper()
    gc_count = seq_str.count('G') + seq_str.count('C')
    total_bases = len(seq_str)
    if total_bases > 0:
        return (gc_count / total_bases) * 100
    return 0.0

gc_content_fwd = calculate_gc_content(primer_fwd)
gc_content_rev = calculate_gc_content(primer_rev)
print(f"\nGC Content - Forward Primer: {gc_content_fwd:.2f}%")
print(f"GC Content - Reverse Primer: {gc_content_rev:.2f}%")

# 3. Calculate Melting Temperature (Tm) using Nearest Neighbor method
# Default parameters are often suitable, but can be adjusted (e.g., salt concentration)
tm_fwd = Tm_NN(primer_fwd)
tm_rev = Tm_NN(primer_rev)
print(f"\nMelting Temperature (Tm) - Forward Primer: {tm_fwd:.2f} °C")
print(f"Melting Temperature (Tm) - Reverse Primer: {tm_rev:.2f} °C")

# 4. Basic Check for Secondary Structures (Self-complementarity for potential hairpins)
# This is a conceptual check; true secondary structure prediction is more complex.
def check_self_complementarity(primer_seq, min_complementary_len=4):
    rc_primer = primer_seq.reverse_complement()
    complementary_regions = []
    for i in range(len(primer_seq) - min_complementary_len + 1):
        for j in range(i + min_complementary_len, len(primer_seq) + 1):
            sub_primer = primer_seq[i:j]
            if sub_primer == rc_primer[len(primer_seq)-j:len(primer_seq)-i]:
                complementary_regions.append(sub_primer)
    return complementary_regions

# A simpler check: finding short complementary regions that could form hairpins
def find_hairpin_potential(seq, min_len=4, max_spacer=10):
    potential_hairpins = []
    for i in range(len(seq) - min_len):
        for j in range(i + min_len, len(seq) - min_len):
            stem1 = seq[i : i + min_len]
            stem2 = seq[j : j + min_len].reverse_complement()
            if stem1 == stem2 and (j - (i + min_len)) <= max_spacer:
                potential_hairpins.append(f"Stem: {stem1} at {i}-{i+min_len}, Loop: {seq[i+min_len:j]} at {i+min_len}-{j}, Complementary Stem: {stem2.reverse_complement()} at {j}-{j+min_len}")
    return potential_hairpins


print(f"\n--- Secondary Structure Check ---")
hairpins_fwd = find_hairpin_potential(primer_fwd)
hairpins_rev = find_hairpin_potential(primer_rev)

if hairpins_fwd:
    print(f"Forward Primer potential hairpins: {hairpins_fwd}")
else:
    print(f"Forward Primer: No significant hairpins detected (basic check).")

if hairpins_rev:
    print(f"Reverse Primer potential hairpins: {hairpins_rev}")
else:
    print(f"Reverse Primer: No significant hairpins detected (basic check).")


# 5. Conceptual In-silico PCR Simulation
# This is a very basic simulation. Real-world PCR simulation is complex.
print(f"\n--- In-silico PCR Simulation ---")

# Search for forward primer binding site on the original sequence
fwd_start = insulin_dna_sequence.find(primer_fwd)

# Search for reverse primer binding site on the reverse complement of the original sequence
# Or, find the reverse primer's reverse complement on the original sequence.
rev_comp_primer_rev = primer_rev.reverse_complement()
rev_start = insulin_dna_sequence.find(rev_comp_primer_rev)

if fwd_start != -1 and rev_start != -1 and rev_start > fwd_start:
    product_size = rev_start + len(rev_comp_primer_rev) - fwd_start
    print(f"Primers bind to template (conceptual): Yes")
    print(f"Forward primer binds at: {fwd_start}")
    print(f"Reverse primer (RC) binds at: {rev_start}")
    print(f"Predicted PCR Product Size: {product_size} bp")
else:
    print(f"Primers bind to template (conceptual): No (or in incorrect orientation/order).")


--- Primer Handling for Insulin DNA Sequence ---

Forward Primer: ATGCCTGCCCGGCCGGGATGCAGTGCA
Reverse Primer: TGCATGCCTGCTGTGGGCAGGGCTTTA

GC Content - Forward Primer: 70.37%
GC Content - Reverse Primer: 59.26%

Melting Temperature (Tm) - Forward Primer: 71.55 °C
Melting Temperature (Tm) - Reverse Primer: 66.31 °C

--- Secondary Structure Check ---
Forward Primer potential hairpins: ['Stem: CCCG at 7-11, Loop: GC at 11-13, Complementary Stem: CGGG at 13-17', 'Stem: CCGG at 8-12, Loop:  at 12-12, Complementary Stem: CCGG at 12-16']
Reverse Primer potential hairpins: ['Stem: TGCC at 4-8, Loop: TGCTGTG at 8-15, Complementary Stem: GGCA at 15-19', 'Stem: CCTG at 6-10, Loop: CTGTGGG at 10-17, Complementary Stem: CAGG at 17-21', 'Stem: CTGC at 7-11, Loop: TGTGG at 11-16, Complementary Stem: GCAG at 16-20']

--- In-silico PCR Simulation ---
Primers bind to template (conceptual): No (or in incorrect orientation/order).


In [ ]:
import re
from google.colab import files
print("Enter your Genome sequence")
uploaded = files.upload()
seqfile = next(iter(uploaded))
print("The Genome is uploaded to ur colab workspace\nname of the file is:", seqfile)

def read_primer3plus_sequence(filename):
    sequence = ""
    descriptor = ""
    with open(filename, 'r') as f:
        for line in f:
            line = line.strip()
            if line.startswith("Primer3Plus File"): # Assuming first line is always this descriptor
                descriptor = line
            if line.startswith("SEQUENCE="):
                # Extract the sequence part and remove any trailing keywords like ED_REGION=
                seq_part = line.split("SEQUENCE=")[1]
                # Assuming ED_REGION= might be at the end of the sequence line
                if "ED_REGION=" in seq_part:
                    seq_part = seq_part.split("ED_REGION=")[0]
                sequence += seq_part.replace(" ", "") # Remove spaces if any
    return descriptor, sequence

_, seq = read_primer3plus_sequence(seqfile) # Using the new function
print(f"Extracted Sequence Length: {len(seq)}") # Add a print to confirm sequence extraction

def restriction_sites_with_re(seq, recog_seq):
    sites = []
    for site in re.finditer(recog_seq, seq):
        sites.append(site.start())
    return sites

print('HindIII:', restriction_sites_with_re(seq, 'AAGCTT'))
print('EcoRI:  ', restriction_sites_with_re(seq, 'GAATTC'))
print('KpnI:   ', restriction_sites_with_re(seq, 'GGTACC'))

def restriction_sites(seq, recog_seq):
    """Find the indices of all restriction sites in a sequence."""
    sites = []
    for i in range(len(seq) - len(recog_seq) + 1):
        if seq[i:i+len(recog_seq)] == recog_seq:
            sites.append(i)
    return sites

print('HindIII:', restriction_sites_with_re(seq, 'AAGCTT'))
print('EcoRI:  ', restriction_sites_with_re(seq, 'GAATTC'))
print('KpnI:   ', restriction_sites_with_re(seq, 'GGTACC'))

Enter your Genome sequence


Saving Sequence.txt to Sequence.txt
The Genome is uploaded to ur colab workspace
name of the file is: Sequence.txt
Extracted Sequence Length: 0
HindIII: []
EcoRI:   []
KpnI:    []
HindIII: []
EcoRI:   []
KpnI:    []
